# Simple RAG

A simple Retrieval-Augmented Generation pipeline using LangChain.

**Setup:** Add `OPENAI_API_KEY` to `.env` (or use HuggingFace for embeddings).

In [ ]:
import os
from dotenv import load_dotenv

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate


load_dotenv(".env", override=True)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "").strip()
HUGGING_FACE_API_KEY = os.getenv("HF_TOKEN", "").strip()
print(f"OPENAI_API_KEY: {OPENAI_API_KEY[:4]}...{OPENAI_API_KEY[-4:]}") 
print(f"HUGGING_FACE_API_KEY: {HUGGING_FACE_API_KEY[:4]}...{HUGGING_FACE_API_KEY[-4:]}") 

In [ ]:
# load the documents using langchain's DirectoryLoader
loader = DirectoryLoader(
    "docs",
    glob="**/*.md",
    loader_cls=TextLoader
)

documents = loader.load()

print(f"Loaded {len(documents)} documents")
for doc in documents:
    print(doc.metadata)


In [ ]:
#chunk the documents into smaller pieces using RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks")
print("\nSample chunk:\n")
print(chunks[0].page_content)

In [ ]:
# add metadata to the chunks
for chunk in chunks:
    source = chunk.metadata.get("source", "").lower()

    if "refund" in source:
        chunk.metadata["topic"] = "refunds"
    elif "shipping" in source:
        chunk.metadata["topic"] = "shipping"
    elif "security" in source:
        chunk.metadata["topic"] = "security"
    elif "faq" in source:
        chunk.metadata["topic"] = "faq"
    else:
        chunk.metadata["topic"] = "general"

chunks[0].metadata

In [ ]:
# Now we add embeddings using HuggingFaceEmbeddings and store in ChromaDB



embeddings = HuggingFaceEmbeddings(


    model_name="sentence-transformers/all-MiniLM-l6-v2"
)

# quick sanity check
test_vec = embeddings.embed_query("How long do refunds take?")
print(f"Embedding length: {len(test_vec)}")
print(test_vec[:5])

In [ ]:
# we store the embeddings in ChromaDB

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="customer_support_docs",
    persist_directory="./chroma_db"
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
print ("Chroma DB setup complete.")
print(f"Number of documents in ChromaDB: {vectorstore._collection.count()}")


In [ ]:
# test retrieval
query = "How long do refunds take?"

results = retriever.invoke(query)

for i, doc in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print("Topic:", doc.metadata.get("topic"))
    print(doc.page_content)
    

In [ ]:
# add topic classifier for hierachy

def classify_topic(question: str) -> str:
    q = question.lower()
    
    if any(word in q for word in ["refund", "refunds", "money back", "reimbursement"]):
        return "refunds"
    elif any(word in q for word in ["shipping", "delivery", "dispatch", "tracking", "address"]):
        return "shipping"
    elif any(word in q for word in ["security", "login", "password", "locked", "suspicious"]):
        return "security"
    elif any(word in q for word in ["contact", "faq", "question", "support"]):
        return "faq"
    else:
        return "general"

In [ ]:
# Retrive with meta data filtering
def retrieve_with_topic(question: str, k: int = 4):
    topic = classify_topic(question)
    docs = vectorstore.similarity_search(
        question,
        k=k,
        filter={"topic": topic}
    )
    return topic, docs

topic, docs = retrieve_with_topic("How long do refunds take?")
print("Topic:", topic)

for i, d in enumerate(docs, 1):
    print(f"\n--- Doc {i} ---")
    print(d.page_content)

In [ ]:
client = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

rewrite_prompt = ChatPromptTemplate.from_template("""
Rewrite the user's question into a clearer search query for retrieving relevant support documents.
Keep the meaning the same.
Fix grammar and spelling.
Return only the rewritten query.

Question: {question}
""")

rewrite_chain = rewrite_prompt | client | StrOutputParser()

test_question = "when are refsds processed"
rewritten = rewrite_chain.invoke({"question": test_question})

print("Original:", test_question)
print("Rewritten:", rewritten)



In [ ]:
expand_prompt = ChatPromptTemplate.from_template("""
Generate 3 alternative search queries for the user question below.
They should preserve meaning but use different wording.
Return each query on a new line only.

Question: {question}
""")

expand_chain = expand_prompt | client | StrOutputParser()

def expand_query(question: str) -> list[str]:
    raw = expand_chain.invoke({"question": question})
    lines = [line.strip("-• ").strip() for line in raw.split("\n") if line.strip()]
    unique = []
    seen = set()
    for line in lines:
        if line not in seen:
            seen.add(line)
            unique.append(line)
    return unique[:3]

expand_query("How long do refunds take?")

In [ ]:

# Combine everything together: rewrite, expand, retrieve with filtering, and deduplication
def retrieve_expanded(question: str, k_per_query: int = 3):
    rewritten = rewrite_chain.invoke({"question": question})
    expanded = expand_query(rewritten)
    all_queries = [rewritten] + expanded

    all_docs = []
    seen_text = set()

    for q in all_queries:
        topic, docs = retrieve_with_topic(q, k=k_per_query)
        for doc in docs:
            key = doc.page_content.strip()
            if key not in seen_text:
                seen_text.add(key)
                all_docs.append(doc)

    return {
        "rewritten_query": rewritten,
        "expanded_queries": expanded,
        "documents": all_docs
    }

In [ ]:

# reranking with LLMs
score_prompt = ChatPromptTemplate.from_template("""
Given the user question and a retrieved document chunk, score how relevant the chunk is from 1 to 10.

Return only the number.

Question: {question}

Chunk:
{chunk}
""")

score_chain = score_prompt | client | StrOutputParser()

def score_doc(question: str, chunk: str) -> float:
    raw = score_chain.invoke({"question": question, "chunk": chunk}).strip()
    try:
        return float(raw)
    except:
        return 0.0

In [ ]:
# keep the best chunks
def rerank_documents(question: str, docs: list, top_n: int = 3):
    scored = []
    for doc in docs:
        score = score_doc(question, doc.page_content)
        scored.append((score, doc))

    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[:top_n]

In [ ]:
def format_context(reranked_docs):
    return "\n\n".join([doc.page_content for score, doc in reranked_docs])

In [ ]:
# final answer generation chain
answer_prompt = ChatPromptTemplate.from_template("""
You are a helpful support assistant.
Answer the user's question using only the context below.
If the answer is not in the context, say "I don't know."

Context:
{context}

Question:
{question}
""")

answer_chain = answer_prompt | client | StrOutputParser()

def answer_question(question: str):
    retrieved = retrieve_expanded(question)
    reranked = rerank_documents(question, retrieved["documents"], top_n=3)
    context = format_context(reranked)
    answer = answer_chain.invoke({"context": context, "question": question})

    return {
        "rewritten_query": retrieved["rewritten_query"],
        "expanded_queries": retrieved["expanded_queries"],
        "reranked_docs": reranked,
        "context": context,
        "answer": answer
    }

result = answer_question("How long do refunds take?")
print(result["answer"])

In [ ]:
def router(question: str):
    q = question.lower()

    return {
        "use_rewrite": True,
        "use_expansion": True if len(q.split()) < 8 else False,
        "use_topic_filter": True,
        "use_rerank": True
    }

In [ ]:
# agentic pipeline
def agentic_rag(question: str):
    decisions = router(question)

    working_query = question
    expanded_queries = [question]

    if decisions["use_rewrite"]:
        working_query = rewrite_chain.invoke({"question": question})

    if decisions["use_expansion"]:
        expanded_queries = [working_query] + expand_query(working_query)
    else:
        expanded_queries = [working_query]

    all_docs = []
    seen = set()

    for q in expanded_queries:
        if decisions["use_topic_filter"]:
            _, docs = retrieve_with_topic(q, k=3)
        else:
            docs = retriever.invoke(q)

        for d in docs:
            key = d.page_content.strip()
            if key not in seen:
                seen.add(key)
                all_docs.append(d)

    if decisions["use_rerank"]:
        reranked = rerank_documents(question, all_docs, top_n=3)
    else:
        reranked = [(0, d) for d in all_docs[:3]]

    context = format_context(reranked)
    answer = answer_chain.invoke({"context": context, "question": question})

    return {
        "decisions": decisions,
        "rewritten_query": working_query,
        "expanded_queries": expanded_queries,
        "context": context,
        "answer": answer,
        "sources": reranked
    }

In [ ]:
# gradio ui

import gradio as gr

def gradio_rag(question):
    result = agentic_rag(question)

    sources_text = []
    for i, (score, doc) in enumerate(result["sources"], 1):
        sources_text.append(
            f"Source {i} | score={score}\n"
            f"topic={doc.metadata.get('topic')}\n"
            f"{doc.page_content}"
        )

    return (
        result["answer"],
        result["rewritten_query"],
        "\n".join(result["expanded_queries"]),
        "\n\n---\n\n".join(sources_text),
        str(result["decisions"])
    )

demo = gr.Interface(
    fn=gradio_rag,
    inputs=gr.Textbox(label="Ask a support question", lines=2),
    outputs=[
        gr.Textbox(label="Answer"),
        gr.Textbox(label="Rewritten Query"),
        gr.Textbox(label="Expanded Queries"),
        gr.Textbox(label="Retrieved Sources"),
        gr.Textbox(label="Agent Decisions")
    ],
    title="Support Knowledge RAG Assistant",
    description="LangChain + Hugging Face embeddings + Chroma + ChatOpenAI + Gradio"
)

demo.launch()